In [2]:
# ─────────────────────────────────────────────────────────
# CELL 0: Environment Setup & Imports
# ─────────────────────────────────────────────────────────
import os, sys, gc
import numpy as np
import torch

# Print available input files
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Make pipeline.py importable — adjust path if running on Kaggle
# When the notebook is committed on Kaggle, pipeline.py must be
# uploaded as part of the notebook's utility scripts or placed in
# the working directory.
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

from pipeline import (
    Config, seed_everything,
    compute_grid_stats, save_stats, load_stats,
    build_samples, build_test_inputs,
    make_dataloaders,
    build_model, train_model,
    run_inference, save_predictions,
    average_domain_rmse, run_full_pipeline,
)

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

/kaggle/input/competitions/aisehack-theme-2/test_in/psfc.npy
/kaggle/input/competitions/aisehack-theme-2/test_in/t2.npy
/kaggle/input/competitions/aisehack-theme-2/test_in/SO2.npy
/kaggle/input/competitions/aisehack-theme-2/test_in/NMVOC_finn.npy
/kaggle/input/competitions/aisehack-theme-2/test_in/bio.npy
/kaggle/input/competitions/aisehack-theme-2/test_in/rain.npy
/kaggle/input/competitions/aisehack-theme-2/test_in/u10.npy
/kaggle/input/competitions/aisehack-theme-2/test_in/swdown.npy
/kaggle/input/competitions/aisehack-theme-2/test_in/pblh.npy
/kaggle/input/competitions/aisehack-theme-2/test_in/v10.npy
/kaggle/input/competitions/aisehack-theme-2/test_in/cpm25.npy
/kaggle/input/competitions/aisehack-theme-2/test_in/NH3.npy
/kaggle/input/competitions/aisehack-theme-2/test_in/q2.npy
/kaggle/input/competitions/aisehack-theme-2/test_in/NMVOC_e.npy
/kaggle/input/competitions/aisehack-theme-2/test_in/NOx.npy
/kaggle/input/competitions/aisehack-theme-2/test_in/PM25.npy
/kaggle/input/competit

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 1: Configuration
# ─────────────────────────────────────────────────────────
# All hyper-parameters are centralised in Config.
# Override any defaults here — the rest of the pipeline adapts automatically.

cfg = Config(
    # ── Paths ──
    DATA_DIR="/kaggle/input/competitions/aisehack-theme-2",
    TEMP_DIR="/kaggle/temp",
    WORK_DIR="/kaggle/working",

    # ── Feature selection ──
    # Using all 8 met + 7 emis features (change to experiment)
    MET_FEATURES=["u10", "v10", "pblh", "rain", "t2", "q2", "swdown", "psfc"],
    EMIS_FEATURES=["PM25", "SO2", "NOx", "NH3", "NMVOC_e", "NMVOC_finn", "bio"],

    # ── Temporal ──
    T_IN_CPM=10,
    T_IN_MET=26,
    T_OUT=16,

    # ── Train / Val split ──
    ALL_MONTHS=["APRIL_16", "JULY_16", "OCT_16", "DEC_16"],
    VAL_MONTH="OCT_16",
    TRAIN_STRIDE=2,
    VAL_STRIDE=4,

    # ── Training ──
    BATCH_SIZE_TRAIN=4,
    BATCH_SIZE_VAL=8,
    BATCH_SIZE_TEST=16,
    NUM_WORKERS=2,
    EPOCHS=30,
    LR=1e-3,
    WEIGHT_DECAY=1e-4,
    GRAD_CLIP=1.0,
    PCT_START=0.1,

    # ── Model ──
    FNO_WIDTH=64,
    FNO_MODES=20,
    FNO_DEPTH=4,

    SEED=42,
)

seed_everything(cfg.SEED)
print(cfg)

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 2: Compute Grid-wise Normalization Statistics
# ─────────────────────────────────────────────────────────
# Per-gridpoint mean/std computed ONLY on training months
# to avoid data leakage from the validation month.

stats = compute_grid_stats(
    data_dir=cfg.DATA_DIR,
    months=cfg.train_months,
    features=cfg.features,
)

stats_path = os.path.join(cfg.TEMP_DIR, "norm_stats.npy")
os.makedirs(cfg.TEMP_DIR, exist_ok=True)
save_stats(stats, stats_path)

print(f"Stats computed for {len(cfg.features)} features across {cfg.train_months}")
print(f"Example — cpm25 mean range: [{stats['cpm25']['mean'].min():.2f}, {stats['cpm25']['mean'].max():.2f}]")

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 3: Build Train and Validation Samples
# ─────────────────────────────────────────────────────────
# Sliding-window samples:
#   X: (N, T_IN_MET, 140, 124, C)  — normalised, cpm25 masked after 10h
#   y: (N, 140, 124, T_OUT)        — raw µg/m³ ground truth

X_train, y_train = build_samples(
    data_dir=cfg.DATA_DIR,
    months=cfg.train_months,
    features=cfg.features,
    stats=stats,
    t_in_cpm=cfg.T_IN_CPM,
    t_in_met=cfg.T_IN_MET,
    t_out=cfg.T_OUT,
    stride=cfg.TRAIN_STRIDE,
)

X_val, y_val = build_samples(
    data_dir=cfg.DATA_DIR,
    months=[cfg.VAL_MONTH],
    features=cfg.features,
    stats=stats,
    t_in_cpm=cfg.T_IN_CPM,
    t_in_met=cfg.T_IN_MET,
    t_out=cfg.T_OUT,
    stride=cfg.VAL_STRIDE,
)

print(f"Train — X: {X_train.shape}, y: {y_train.shape}")
print(f"Val   — X: {X_val.shape},   y: {y_val.shape}")
print(f"Features ({cfg.n_features}): {cfg.features}")

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 4: Create Dataset & DataLoaders
# ─────────────────────────────────────────────────────────

train_dl, val_dl = make_dataloaders(
    X_train, y_train,
    X_val,   y_val,
    batch_train=cfg.BATCH_SIZE_TRAIN,
    batch_val=cfg.BATCH_SIZE_VAL,
    num_workers=cfg.NUM_WORKERS,
)

# Free raw numpy arrays — data is now held as tensors inside datasets
del X_train, y_train, X_val, y_val
gc.collect()

print(f"Train batches: {len(train_dl)}, Val batches: {len(val_dl)}")

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 5: Build TFNO2D Model
# ─────────────────────────────────────────────────────────
# Improved FNO2D with residual skip connections,
# deeper projection head, and dropout regularisation.

model = build_model(cfg)

# Quick shape sanity check (dry run)
dummy = torch.randn(1, cfg.n_features, cfg.T_IN_MET, 140, 124, device=cfg.DEVICE)
out = model(dummy)
print(f"Dry-run output: {out.shape}  (expected: (1, 140, 124, {cfg.T_OUT}))")
del dummy, out
torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 6: Train the Model
# ─────────────────────────────────────────────────────────
# Uses metric-aligned loss (average domain RMSE), OneCycleLR
# scheduler, gradient clipping, and best-checkpoint saving.

model = train_model(model, train_dl, val_dl, cfg)

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 7: Prepare Test Inputs & Run Inference
# ─────────────────────────────────────────────────────────

# Build normalised test tensor (cpm25 zero-padded from 10→26 steps)
X_test = build_test_inputs(
    data_dir=cfg.DATA_DIR,
    features=cfg.features,
    stats=stats,
    t_in_cpm=cfg.T_IN_CPM,
    t_in_met=cfg.T_IN_MET,
)
print(f"Test input shape: {X_test.shape}")

# Run batched inference → denormalise → clamp ≥ 0
preds = run_inference(
    model=model,
    X_test=X_test,
    stats=stats,
    target_feat=cfg.TARGET,
    device=cfg.DEVICE,
    batch_size=cfg.BATCH_SIZE_TEST,
)

# Save to /kaggle/working/preds.npy
save_predictions(preds, cfg.preds_path)

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 8: Quick Prediction Sanity Checks
# ─────────────────────────────────────────────────────────
preds_loaded = np.load(cfg.preds_path)
print(f"Shape      : {preds_loaded.shape}")
print(f"dtype      : {preds_loaded.dtype}")
print(f"All finite : {np.isfinite(preds_loaded).all()}")
print(f"Min        : {preds_loaded.min():.4f}")
print(f"Max        : {preds_loaded.max():.4f}")
print(f"Mean       : {preds_loaded.mean():.4f}")
print(f"Std        : {preds_loaded.std():.4f}")

# Per-hour statistics
for t in range(preds_loaded.shape[-1]):
    hour_data = preds_loaded[:, :, :, t]
    print(f"  Hour {t+1:2d}: mean={hour_data.mean():.2f}  std={hour_data.std():.2f}  "
          f"max={hour_data.max():.2f}")

In [ ]:
# Pipeline Architecture Summary

## Module: `pipeline.py`

| Section | Function / Class | Purpose |
|---------|-----------------|---------|
| **Config** | `Config` | Central config object — all hyper-params in one place |
| **Seed** | `seed_everything()` | Reproducibility (numpy + torch + CUDA) |
| **Data I/O** | `load_raw_feature()`, `load_test_feature()` | Load `.npy` arrays |
| **Normalization** | `compute_grid_stats()` | Per-gridpoint mean/std from train months |
| | `normalize_array()`, `denormalize_array()` | Z-score transforms |
| **Samples** | `build_samples()` | Sliding-window train/val construction |
| | `build_test_inputs()` | Test tensor prep with cpm25 zero-padding |
| **Dataset** | `PM25Dataset`, `make_dataloaders()` | PyTorch DataLoader creation |
| **Model** | `SpectralConv2d`, `FNOBlock`, `TFNO2D` | FNO2D with residual skip connections |
| | `build_model()` | Instantiate from config |
| **Training** | `metric_loss()` | Competition-aligned RMSE loss |
| | `train_one_epoch()`, `validate()`, `train_model()` | Full training loop |
| **Inference** | `run_inference()`, `save_predictions()` | Batched prediction + denorm + save |
| **Evaluation** | `average_domain_rmse()` | Offline metric calculation |
| **E2E** | `run_full_pipeline()` | One-call end-to-end runner |